# 04 — Insights Finais

Este notebook consolida os principais achados dos notebooks anteriores em uma narrativa objetiva sobre **desigualdades educacionais no Brasil**, usando os microdados do ENEM 2023.

**Pergunta central:** quais fatores estão mais associados ao desempenho no ENEM — e qual a magnitude dessas diferenças?

**Seções:**
1. Os 5 principais insights
2. Amplitude das desigualdades (gráfico síntese)
3. A desvantagem composta
4. Conclusão

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sys.path.insert(0, '../src')
from utils import NOTAS_COLS, NOTAS_LABELS, configurar_estilo, salvar_figura

configurar_estilo()

df = pd.read_parquet('../data/processed/amostra_limpa.parquet')

REGIOES = {
    'Norte':        ['AM', 'PA', 'AC', 'RO', 'RR', 'AP', 'TO'],
    'Nordeste':     ['MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA'],
    'Centro-Oeste': ['MT', 'MS', 'GO', 'DF'],
    'Sudeste':      ['SP', 'RJ', 'MG', 'ES'],
    'Sul':          ['PR', 'SC', 'RS'],
}
df['REGIAO'] = df['SG_UF_PROVA'].map(
    {uf: reg for reg, ufs in REGIOES.items() for uf in ufs}
)

print(f'Candidatos na amostra: {len(df):,}'.replace(',', '.'))
print(f'Nota média geral:      {df["NOTA_MEDIA"].mean():.1f} pts')

## 1. Os 5 Principais Insights

| # | Insight | Evidência |
|---|---|---|
| 1 | **Escola privada supera pública em todas as áreas** | +101 pts na média; gap mínimo de 45 pts em qualquer faixa de renda |
| 2 | **Renda é o fator de maior amplitude absoluta** | +172 pts entre a faixa sem renda e a mais alta |
| 3 | **Desigualdades raciais são sistemáticas** | Indígenas: −82 pts vs brancos; pretos: −52 pts; pardos: −46 pts |
| 4 | **Diferenças de gênero são área-específicas** | Homens +38 pts em MT; mulheres +39 pts em Redação |
| 5 | **Norte e Nordeste seguem em desvantagem regional** | Mediana do Sul/Sudeste ~52 pts acima do Norte |

> **O achado mais relevante:** a vantagem da escola privada persiste em **todas** as 16 faixas de renda (gap mínimo de 45 pts, máximo de 87 pts), sugerindo que o tipo de escola tem efeito independente da renda familiar.

## 2. Amplitude das Desigualdades

O gráfico abaixo compara, em uma única escala, a **diferença de nota média entre os grupos extremos** de cada dimensão analisada.

In [ ]:
renda_m  = df.groupby('Q006', observed=True)['NOTA_MEDIA'].mean()
escola_m = (df[df['TP_ESCOLA'].isin(['Privada', 'Pública'])]
            .groupby('TP_ESCOLA', observed=True)['NOTA_MEDIA'].mean())
raca_m   = df.groupby('TP_COR_RACA', observed=True)['NOTA_MEDIA'].mean()
regiao_m = df.groupby('REGIAO', observed=True)['NOTA_MEDIA'].median()
mt_m     = df.groupby('TP_SEXO', observed=True)['NU_NOTA_MT'].mean()
red_m    = df.groupby('TP_SEXO', observed=True)['NU_NOTA_REDACAO'].mean()

gaps = pd.Series({
    'Renda\n(sem renda → acima R$ 39k)':      renda_m.max() - renda_m.min(),
    'Tipo de escola\n(Privada → Pública)':     escola_m['Privada'] - escola_m['Pública'],
    'Raça\n(Branca → Indígena)':               raca_m['Branca'] - raca_m['Indígena'],
    'Raça\n(Branca → Preta)':                  raca_m['Branca'] - raca_m['Preta'],
    'Região\n(Sul/Sudeste → Norte)':            regiao_m[['Sul', 'Sudeste']].mean() - regiao_m['Norte'],
    'Raça\n(Branca → Parda)':                  raca_m['Branca'] - raca_m['Parda'],
    'Gênero — Redação\n(Feminino → Masculino)': red_m['Feminino'] - red_m['Masculino'],
    'Gênero — MT\n(Masculino → Feminino)':      mt_m['Masculino'] - mt_m['Feminino'],
}).sort_values()

PALETA = {
    'Renda':  '#e74c3c',
    'Escola': "#eac54c",
    'Raça':   '#e67e22',
    'Região': '#2980b9',
    'Gênero': '#27ae60',
}

def cor(label):
    for k, c in PALETA.items():
        if k in label:
            return c
    return '#eac54c'

cores = [cor(k) for k in gaps.index]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(gaps.index, gaps.values, color=cores, alpha=0.85, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, gaps.values):
    ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
            f'+{val:.0f} pts', va='center', fontsize=9)

ax.set_xlabel('Diferença de nota média (pts)', fontsize=11)
ax.set_title(
    'Amplitude das Desigualdades no ENEM 2023\n'
    'Diferença entre o grupo de maior e menor desempenho em cada dimensão',
    fontsize=12, pad=12
)
ax.set_xlim(0, gaps.max() + 30)
ax.axvline(0, color='black', linewidth=0.8)

legendas = [mpatches.Patch(color=c, label=l) for l, c in PALETA.items()]
ax.legend(handles=legendas, loc='lower right', fontsize=9, title='Dimensão')

plt.tight_layout()
salvar_figura(fig, '04_mapa_desigualdades')
plt.show()

## 3. A Desvantagem Composta

As desigualdades se somam: um candidato que acumula múltiplos fatores desfavoráveis parte de uma posição muito distante dos grupos privilegiados.

O gráfico abaixo compara a nota média de **perfis extremos** — combinações de escola, renda e cor/raça.

In [ ]:
renda_baixa = renda_m.idxmin()
renda_alta  = renda_m.idxmax()

perfis = {
    'Escola pública\nRenda baixa\nCor/raça Indígena': df[
        (df['TP_ESCOLA'] == 'Pública') &
        (df['Q006'] == renda_baixa) &
        (df['TP_COR_RACA'] == 'Indígena')
    ]['NOTA_MEDIA'].mean(),

    'Escola pública\nRenda baixa\nCor/raça Preta': df[
        (df['TP_ESCOLA'] == 'Pública') &
        (df['Q006'] == renda_baixa) &
        (df['TP_COR_RACA'] == 'Preta')
    ]['NOTA_MEDIA'].mean(),

    'Média geral': df['NOTA_MEDIA'].mean(),

    'Escola privada\nRenda alta\nCor/raça Branca': df[
        (df['TP_ESCOLA'] == 'Privada') &
        (df['Q006'] == renda_alta) &
        (df['TP_COR_RACA'] == 'Branca')
    ]['NOTA_MEDIA'].mean(),
}

perfis_s = pd.Series(perfis).dropna().sort_values()

CORES_PERFIL = [
    '#e74c3c' if 'pública' in k.lower() else
    '#95a5a6' if 'Média' in k else
    '#27ae60'
    for k in perfis_s.index
]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(perfis_s.index, perfis_s.values, color=CORES_PERFIL, alpha=0.85,
               edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, perfis_s.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}', va='center', fontsize=9)

gap_total = perfis_s.max() - perfis_s.min()
ax.set_xlabel('Nota Média', fontsize=11)
ax.set_title(
    f'Desvantagem Composta — ENEM 2023\nGap entre perfis extremos: {gap_total:.0f} pts',
    fontsize=12, pad=12
)
ax.set_xlim(perfis_s.min() - 20, perfis_s.max() + 30)

legendas = [
    mpatches.Patch(color='#e74c3c', label='Perfil desfavorecido'),
    mpatches.Patch(color='#95a5a6', label='Média geral'),
    mpatches.Patch(color='#27ae60', label='Perfil privilegiado'),
]
ax.legend(handles=legendas, loc='lower right', fontsize=9)

plt.tight_layout()
salvar_figura(fig, '04_desvantagem_composta')
plt.show()

print(f'Gap total entre perfis extremos: {gap_total:.0f} pts')

## 4. Conclusão

Os dados do ENEM 2023 revelam desigualdades educacionais profundas e sistemáticas no Brasil:

- **Renda** é o fator de maior amplitude isolada (+172 pts), mas opera em conjunto com todos os outros.
- **Tipo de escola** tem efeito independente da renda — a escola privada supera a pública em todas as 16 faixas de renda investigadas.
- **Raça e região** acumulam desvantagem: candidatos indígenas, pretos e pardos, em sua maioria de escolas públicas e regiões Norte/Nordeste, concentram múltiplos fatores desfavoráveis.
- **Gênero** apresenta diferenças específicas por área: homens lideram em Matemática, mulheres em Redação — sem vantagem geral de nenhum grupo.

O achado mais importante é a **desvantagem composta**: quando os fatores se somam (escola pública + renda baixa + cor/raça não branca), o gap ultrapassa 200 pts em relação ao perfil mais privilegiado — equivalente a mais de dois desvios padrão da distribuição de notas.